# Computational Models Lab: Q-Learning on Concurrent VI-VI Schedules

In this lab, you will build a reinforcement learning agent from scratch and test whether a simple Q-learning algorithm produces choice allocation that approximates the **matching law** on concurrent variable-interval (VI) schedules.

Recall from Week 2 that the generalized matching equation predicts:

$$\frac{B_1}{B_2} = b \left(\frac{r_1}{r_2}\right)^a$$

where $B$ is behavior allocation, $r$ is reinforcement rate, $a$ is sensitivity to reinforcement, and $b$ is bias.

Here we ask: does an agent that learns purely from trial-and-error via Q-value updates converge on a steady-state allocation that looks like matching?

**Objectives:**
1. Implement a concurrent VI 30-s vs VI 60-s schedule environment
2. Implement the Q-learning update rule
3. Run simulations and analyze steady-state choice allocation
4. Sweep learning rate and observe its effect on convergence
5. Compare agent behavior to generalized matching predictions

## Setup

Run the cell below to import the libraries you will need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Task 1: Define the Concurrent VI-VI Schedule Environment

A variable-interval (VI) schedule arranges reinforcement for the first response after a variable amount of time has elapsed. On a **concurrent** VI-VI schedule, the organism can allocate responses to two alternatives, each operating on its own VI timer.

Implement a `ConcurrentVIVI` class with the following behavior:

- **Two VI timers**: one for the left alternative (VI 30 s) and one for the right (VI 60 s).
- Each timer samples inter-reinforcement intervals from an **exponential distribution** with the appropriate mean.
- At each time step (1 second), the timers tick down. If a timer has elapsed and the agent responds on that alternative, a reinforcer is delivered and the timer resets.
- The environment should track: total responses to each alternative, total reinforcers from each alternative.

The class needs:
- `__init__(self, vi_left=30, vi_right=60)` 
- `reset(self)` -> returns initial state
- `step(self, action)` -> returns `(next_state, reward, done)` where action is 0 (left) or 1 (right)

Since concurrent VI schedules are stationary (no state transitions), you can use a single state. The episode ends after a fixed session duration (e.g., 1800 seconds = 30 min).

In [ ]:
class ConcurrentVIVI:
    """Concurrent VI-VI schedule environment."""

    def __init__(self, vi_left=30, vi_right=60, session_duration=1800):
        self.vi_left = vi_left
        self.vi_right = vi_right
        self.session_duration = session_duration

    def reset(self):
        """Reset the environment for a new session. Return initial state."""
        self.t = 0
        # Time (s) until each schedule next sets up a reinforcer.
        self.timer_left = np.random.exponential(self.vi_left)
        self.timer_right = np.random.exponential(self.vi_right)
        self.avail_left = False
        self.avail_right = False
        self.responses = [0, 0]
        self.reinforcers = [0, 0]
        return 0

    def step(self, action):
        self.t += 1
        # VI timers "hold": they only run while no reinforcer is already set up.
        if not self.avail_left:
            self.timer_left -= 1
            if self.timer_left <= 0:
                self.avail_left = True
                self.timer_left = np.random.exponential(self.vi_left)
        if not self.avail_right:
            self.timer_right -= 1
            if self.timer_right <= 0:
                self.avail_right = True
                self.timer_right = np.random.exponential(self.vi_right)

        # The agent emits one response on the chosen alternative this second.
        self.responses[action] += 1
        reward = 0.0
        if action == 0 and self.avail_left:
            reward = 1.0
            self.avail_left = False
            self.reinforcers[0] += 1
        elif action == 1 and self.avail_right:
            reward = 1.0
            self.avail_right = False
            self.reinforcers[1] += 1

        done = self.t >= self.session_duration
        return 0, reward, done

## Task 2: Implement the Q-Learning Agent

Implement a Q-learning agent. Because our environment has only one state and two actions, the Q-table is simply a 1x2 array (or just a length-2 vector).

The Q-value update rule is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

where:
- $\alpha$ is the learning rate
- $\gamma$ is the discount factor
- $r$ is the reward received
- $s'$ is the next state

For action selection, use a **softmax** policy:

$$P(a) = \frac{e^{Q(s,a) / \tau}}{\sum_{a'} e^{Q(s,a') / \tau}}$$

where $\tau$ (tau) is a temperature parameter controlling exploration vs exploitation.

Implement the `QLearningAgent` class with:
- `__init__(self, n_actions=2, alpha=0.1, gamma=0.95, tau=0.5)`
- `select_action(self, state)` -> action (using softmax)
- `update(self, state, action, reward, next_state)` -> updates Q-values

In [ ]:
class QLearningAgent:
    """Q-learning agent with softmax action selection."""

    def __init__(self, n_actions=2, alpha=0.1, gamma=0.95, tau=0.5):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.tau = tau
        self.Q = np.zeros(n_actions)

    def select_action(self, state):
        """Select an action using a softmax policy."""
        z = self.Q / self.tau
        z = z - np.max(z)            # subtract max for numerical stability
        p = np.exp(z)
        p = p / p.sum()
        return np.random.choice(self.n_actions, p=p)

    def update(self, state, action, reward, next_state):
        """Update Q-values using the Q-learning rule (single state)."""
        best_next = np.max(self.Q)
        td_target = reward + self.gamma * best_next
        self.Q[action] += self.alpha * (td_target - self.Q[action])

## Task 3: Run a Single Session

Write a function `run_session(agent, env)` that:
1. Resets the environment
2. At each time step, the agent selects an action, the environment returns a reward, and the agent updates its Q-values
3. Returns the total responses and reinforcers for each alternative

Test it by running one session with default parameters and printing the results.

In [ ]:
def run_session(agent, env):
    """Run one session and return choice and reinforcement counts."""
    state = env.reset()
    done = False
    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state)
        state = next_state
    return {
        'responses_left': env.responses[0],
        'responses_right': env.responses[1],
        'reinforcers_left': env.reinforcers[0],
        'reinforcers_right': env.reinforcers[1],
        'q_values': agent.Q.copy(),
    }


# Test with one session
env = ConcurrentVIVI()
agent = QLearningAgent()
result = run_session(agent, env)
print(result)

## Task 4: Run Multiple Sessions and Track Learning

Now run the agent for **50 sessions** (the Q-values persist across sessions, simulating an organism learning over days).

For each session, record:
- The proportion of responses allocated to the left (richer) alternative: $B_L / (B_L + B_R)$
- The proportion of reinforcers obtained from the left alternative: $r_L / (r_L + r_R)$
- The Q-values at the end of each session

Plot:
1. **Learning curve**: Choice proportion (left) across sessions
2. **Q-values**: Q(left) and Q(right) across sessions
3. Add a horizontal dashed line at the **matching prediction** (for VI 30 vs VI 60, the matching law predicts $B_L / (B_L + B_R) = 0.667$)

In [ ]:
n_sessions = 50
env = ConcurrentVIVI(vi_left=30, vi_right=60)
agent = QLearningAgent(alpha=0.1, gamma=0.95, tau=0.5)

choice_prop, reinf_prop, q_left, q_right = [], [], [], []
for s in range(n_sessions):
    res = run_session(agent, env)
    bl, br = res['responses_left'], res['responses_right']
    rl, rr = res['reinforcers_left'], res['reinforcers_right']
    choice_prop.append(bl / (bl + br))
    reinf_prop.append(rl / (rl + rr) if (rl + rr) > 0 else np.nan)
    q_left.append(res['q_values'][0])
    q_right.append(res['q_values'][1])

# Matching prediction for VI 30 vs VI 60: rate_L / (rate_L + rate_R)
matching_pred = (1 / 30) / (1 / 30 + 1 / 60)  # = 0.667

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(choice_prop, label='Choice proportion (left)')
axes[0].plot(reinf_prop, label='Reinforcer proportion (left)', alpha=0.6)
axes[0].axhline(matching_pred, ls='--', color='k',
                label=f'Matching prediction ({matching_pred:.3f})')
axes[0].set_xlabel('Session'); axes[0].set_ylabel('Proportion (left)')
axes[0].set_title('Learning curve'); axes[0].set_ylim(0, 1); axes[0].legend()

axes[1].plot(q_left, label='Q(left)')
axes[1].plot(q_right, label='Q(right)')
axes[1].set_xlabel('Session'); axes[1].set_ylabel('Q-value')
axes[1].set_title('Q-values across sessions'); axes[1].legend()
plt.tight_layout(); plt.show()

## Task 5: Does the Agent Match?

Analyze the agent's steady-state behavior (last 10 sessions):

1. Compute the mean choice proportion for the left alternative over the last 10 sessions.
2. Compute the mean reinforcer proportion for the left alternative over the last 10 sessions.
3. Compare to the strict matching prediction: $B_L / B_R = r_L / r_R$ (i.e., $B_L/(B_L+B_R) \approx 0.667$).
4. Does the agent **undermatch**, **match**, or **overmatch**? 

Print the results and write 2-3 sentences interpreting what you find.

In [ ]:
last10_choice = np.mean(choice_prop[-10:])
last10_reinf = np.nanmean(reinf_prop[-10:])
print(f"Mean choice proportion (left), last 10 sessions: {last10_choice:.3f}")
print(f"Mean reinforcer proportion (left), last 10 sessions: {last10_reinf:.3f}")
print(f"Strict matching prediction: {matching_pred:.3f}")

if last10_choice < matching_pred - 0.02:
    verdict = "undermatching (allocation less extreme than reinforcement)"
elif last10_choice > matching_pred + 0.02:
    verdict = "overmatching (allocation more extreme than reinforcement)"
else:
    verdict = "approximate matching"
print(f"Verdict: the agent shows {verdict}.")

*Write your interpretation here.*

## Task 6: Learning Rate Sweep

The learning rate $\alpha$ controls how quickly the agent updates its value estimates. Sweep across **5 values** of alpha:

$$\alpha \in \{0.01, 0.05, 0.1, 0.3, 0.5\}$$

For each alpha value, run 50 sessions and record the choice proportion for the left alternative across sessions.

Plot all 5 learning curves on the same figure (use different colors and a legend). Add the matching prediction line.

Answer: How does alpha affect (a) the speed of convergence and (b) the stability of steady-state behavior?

In [ ]:
alphas = [0.01, 0.05, 0.1, 0.3, 0.5]
plt.figure(figsize=(10, 6))
for a in alphas:
    env = ConcurrentVIVI(vi_left=30, vi_right=60)
    agent = QLearningAgent(alpha=a, gamma=0.95, tau=0.5)
    cp = []
    for s in range(50):
        res = run_session(agent, env)
        cp.append(res['responses_left'] / (res['responses_left'] + res['responses_right']))
    plt.plot(cp, label=f'alpha = {a}')
plt.axhline(matching_pred, ls='--', color='k', label=f'Matching ({matching_pred:.3f})')
plt.xlabel('Session'); plt.ylabel('Choice proportion (left)')
plt.title('Effect of learning rate on convergence'); plt.ylim(0, 1); plt.legend()
plt.show()

*Write your answer here.*

## Task 7: Discount Factor Sweep

Now hold alpha constant at 0.1 and sweep the discount factor:

$$\gamma \in \{0.0, 0.5, 0.9, 0.95, 0.99\}$$

Since this is a stationary single-state problem, think carefully about what role gamma plays. Does it matter here? Why or why not?

Run the sweep and plot the results.

In [ ]:
gammas = [0.0, 0.5, 0.9, 0.95, 0.99]
plt.figure(figsize=(10, 6))
for gm in gammas:
    env = ConcurrentVIVI(vi_left=30, vi_right=60)
    agent = QLearningAgent(alpha=0.1, gamma=gm, tau=0.5)
    cp = []
    for s in range(50):
        res = run_session(agent, env)
        cp.append(res['responses_left'] / (res['responses_left'] + res['responses_right']))
    plt.plot(cp, label=f'gamma = {gm}')
plt.axhline(matching_pred, ls='--', color='k', label=f'Matching ({matching_pred:.3f})')
plt.xlabel('Session'); plt.ylabel('Choice proportion (left)')
plt.title('Effect of discount factor (single-state, stationary task)')
plt.ylim(0, 1); plt.legend()
plt.show()

*Write your interpretation of gamma's role here.*

## Task 8: Compare to the Generalized Matching Equation

Recall from Week 2 that the generalized matching equation in logarithmic form is:

$$\log\left(\frac{B_1}{B_2}\right) = a \log\left(\frac{r_1}{r_2}\right) + \log(b)$$

To test this more thoroughly, run the Q-learning agent on **multiple concurrent VI-VI schedule pairs**:

| Condition | Left VI (s) | Right VI (s) | Reinforcer ratio (left:right) |
|-----------|-------------|--------------|-------------------------------|
| 1         | 20          | 60           | 3:1                           |
| 2         | 30          | 60           | 2:1                           |
| 3         | 30          | 30           | 1:1                           |
| 4         | 60          | 30           | 1:2                           |
| 5         | 60          | 20           | 1:3                           |

For each condition:
1. Run 50 sessions
2. Compute the steady-state (last 10 sessions) log response ratio and log reinforcer ratio

Then:
1. Plot log(B_L/B_R) vs log(r_L/r_R)
2. Fit a linear regression to get the sensitivity ($a$) and bias ($\log b$) parameters
3. Compare: does $a \approx 1$ (strict matching)? Is $a < 1$ (undermatching)?

In [ ]:
conditions = [
    {'vi_left': 20, 'vi_right': 60},
    {'vi_left': 30, 'vi_right': 60},
    {'vi_left': 30, 'vi_right': 30},
    {'vi_left': 60, 'vi_right': 30},
    {'vi_left': 60, 'vi_right': 20},
]

log_B, log_r = [], []
for cond in conditions:
    env = ConcurrentVIVI(vi_left=cond['vi_left'], vi_right=cond['vi_right'])
    agent = QLearningAgent(alpha=0.1, gamma=0.95, tau=0.5)
    BL = BR = rL = rR = 0
    for s in range(50):
        res = run_session(agent, env)
        if s >= 40:  # accumulate over the last 10 sessions
            BL += res['responses_left']; BR += res['responses_right']
            rL += res['reinforcers_left']; rR += res['reinforcers_right']
    log_B.append(np.log(BL / BR))
    log_r.append(np.log(rL / rR))

log_B = np.array(log_B); log_r = np.array(log_r)
slope, intercept = np.polyfit(log_r, log_B, 1)
print(f"Sensitivity (a):  {slope:.3f}")
print(f"Bias (log b):     {intercept:.3f}")

plt.figure(figsize=(8, 8))
plt.scatter(log_r, log_B, s=80, zorder=3)
xline = np.linspace(log_r.min(), log_r.max(), 100)
plt.plot(xline, slope * xline + intercept, 'r-',
         label=f'Fit: a = {slope:.2f}, log b = {intercept:.2f}')
plt.plot(xline, xline, 'k--', alpha=0.5, label='Strict matching (a = 1)')
plt.axhline(0, color='gray', lw=0.5); plt.axvline(0, color='gray', lw=0.5)
plt.xlabel('log(r_L / r_R)'); plt.ylabel('log(B_L / B_R)')
plt.title('Generalized matching: Q-learning agent'); plt.legend()
plt.show()

*Interpret the sensitivity and bias parameters. How well does Q-learning approximate the generalized matching equation? What does the sensitivity parameter tell us about the agent's behavior?*

## Task 9: Discussion

Write a short discussion (3-5 paragraphs) addressing:

1. **Emergence of matching**: Did the Q-learning agent produce behavior consistent with the matching law? Was it strict matching, undermatching, or overmatching? Why might a simple RL algorithm produce matching-like behavior?

2. **Mechanism vs. description**: The matching law is a *descriptive* model -- it summarizes behavioral regularities. Q-learning is a *process* model -- it specifies a mechanism that generates behavior. What are the advantages and limitations of each approach?

3. **Biological plausibility**: How might Q-learning relate to actual neural mechanisms of reinforcement learning (e.g., dopamine prediction error signaling)? What aspects of real behavior on concurrent schedules would be difficult for this simple Q-learning model to capture (e.g., changeover delay effects, momentary vs. molar matching)?

*Write your discussion here.*